In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import load_amazon_metadata, parse
load_dotenv(override=True)
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
dataset = load_amazon_metadata("Appliances")
print(f"Number of Appliances: {len(dataset):,}")
# Investigate a particular datapoint

dataset[6]

In [ ]:
# What's the most expensive item?
max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except (TypeError, ValueError):
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

In [ ]:
# Load into Item objects if they have a price range $1-$1000 and enough details

items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
print(items[0].full)

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

# Plot the distribution of lengths

plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)

In [ ]:
# Plot the distribution of prices
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
print(items[3].full)

In [ ]:
from pricer.loaders import ItemLoader

# Process each category sequentially so only one large source dataset is open at a time.
# ItemLoader parallelizes the CPU-heavy parsing *within* each category.
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]
# Accumulate only the compact, validated Item objects returned by each loader.
items = []
for category_number, dataset_name in enumerate(dataset_names, start=1):
    # This category-level status complements ItemLoader's per-chunk progress bar.
    print(f"\n[{category_number}/{len(dataset_names)}] Starting {dataset_name}", flush=True)
    loader = ItemLoader(dataset_name)
    # workers=None (the default) lets the loader adapt to CPU, RAM, and dataset size.
    items.extend(loader.load())
    print(f"  Overall: {category_number}/{len(dataset_names)} categories complete; {len(items):,} items collected", flush=True)
print(f"A grand total of {len(items):,} items")
items[1000]

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# Plot the distribution of prices

prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# Just for good measure, let's shuffle the sample again for the final dataset

random.seed(42)
random.shuffle(sample)
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

# Bar chart by category
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')

plt.xticks(rotation=30, ha='right')

# Add value labels on top of each bar
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

# Display the chart
plt.show()

In [ ]:
# Automotive still in the lead, but improved somewhat
# For another perspective, let's look at a pie

plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

# Add a circle at the center to create a donut chart (optional)
# centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
# fig.gca().add_artist(centre_circle)
plt.title('Categories')

# Equal aspect ratio ensures that pie is drawn as a circle
plt.axis('equal')

plt.show()

In [ ]:
# How does the price vary with the character count?

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")

# Add labels and title
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')

# Display the plot
plt.show()

In [ ]:
# How does the price vary with the weight?

ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")

# Add labels and title
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')

# Display the plot
plt.show()

In [ ]:
username = "akshaymgupte87"
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)

In [ ]:
### Completely independent of above since dataset is already on HuggingFace Hub. This section is just to demonstrate how to load it back in and run a local model on it.

In [1]:
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

False

In [13]:
LITE_MODE = False
username = "akshaymgupte87"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

C:\Users\aksha\PycharmProjects\PricerProje\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\datasets--akshaymgupte87--items_raw_full. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  278MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  279MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 820,000 items
title='Did 520ATV298FB 520 ATV2 X-Ring Chain - 98 Links - Gold' price=85.95 category='Automotive' full='Did  520 ATV2 X-Ring Chain - 98 Links - Gold\n[\'Chain Application: OffroadChain Length: 98Chain Type: 520Marketing Color: GoldGreatly increased sealing performance with four sections. Keeps dirt out and lubrication in. Lowest friction of all types of O-rings. Twisting action disperses the pressure and minimizes power loss. Maximum wear resistance. Comes with clip-type connecting link.\']\n[\'Greatly increased sealing performance with four sections\', \'Keeps dirt out and lubrication in\', \'Lowest friction of all types of O-rings\', \'Twisting action disperses the pressure and minimizes power loss\', \'Maximum wear resistance\']\n{"Product Dimensions": "9 x 6 x 1 inches", "Item Weight": "3.49 pounds", "Manufacturer": "D.I.D", "Is Discontinued By Manufacturer": "No", "Date First Available": "November 25, 2018"}' weight=3.49 summary=None prompt=None id=None


In [14]:
items[2].id

In [15]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [16]:
# Reload the local module so this notebook sees edits made to pricer/batch.py
# without requiring a complete kernel restart.
import importlib
import pricer.batch as batch_module

batch_module = importlib.reload(batch_module)
Batch = batch_module.Batch
MODEL = batch_module.MODEL
OLLAMA_HOST = batch_module.OLLAMA_HOST
SYSTEM_PROMPT = batch_module.SYSTEM_PROMPT
ollama_chat_response = batch_module.ollama_chat_response

# Use the same local Qwen model and prompt configuration as the batch pipeline.
print(items[0].full)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full},
]
response = ollama_chat_response(messages)
summary = response["message"]["content"].strip()
items[0].summary = summary

print(f"\nLocal model: {MODEL}")
print(f"Ollama host: {OLLAMA_HOST}")
print(summary)
print(f"\nInput tokens: {response.get('prompt_eval_count', 0):,}")
print(f"Output tokens: {response.get('eval_count', 0):,}")
print("Cost: $0.00 (local inference)")


Did  520 ATV2 X-Ring Chain - 98 Links - Gold
['Chain Application: OffroadChain Length: 98Chain Type: 520Marketing Color: GoldGreatly increased sealing performance with four sections. Keeps dirt out and lubrication in. Lowest friction of all types of O-rings. Twisting action disperses the pressure and minimizes power loss. Maximum wear resistance. Comes with clip-type connecting link.']
['Greatly increased sealing performance with four sections', 'Keeps dirt out and lubrication in', 'Lowest friction of all types of O-rings', 'Twisting action disperses the pressure and minimizes power loss', 'Maximum wear resistance']
{"Product Dimensions": "9 x 6 x 1 inches", "Item Weight": "3.49 pounds", "Manufacturer": "D.I.D", "Is Discontinued By Manufacturer": "No", "Date First Available": "November 25, 2018"}

Local model: qwen3.6:latest
Ollama host: http://127.0.0.1:11434
Title: DID 520 X-Ring Gold Chain - 98 Links
Category: Motorcycle Parts & Accessories
Brand: DID
Description: This gold-colored 

In [17]:
def make_jsonl(item):
    # Ollama runs locally and does not use OpenAI/Groq batch endpoints.
    # Store each prompt in the same simple format used by pricer.batch.
    line = {
        "custom_id": str(item.id),
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
    }
    return json.dumps(line, ensure_ascii=False)
items[0]

<Did 520ATV298FB 520 ATV2 X-Ring Chain - 98 Links - Gold = $85.95>

In [18]:
make_jsonl(items[0])

'{"custom_id": "0", "model": "qwen3.6:latest", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Did  520 ATV2 X-Ring Chain - 98 Links - Gold\\n[\'Chain Application: OffroadChain Length: 98Chain Type: 520Marketing Color: GoldGreatly increased sealing performance with four sections. Keeps dirt out and lubrication in. Lowest friction of all types of O-rings. Twisting action disperses the pressure and minimizes power loss. Maximum wear resistance. Comes with clip-type connecting link.\']\\n[\'Greatly increased sealing performance with four sections\', \'Keeps dirt out and lubrication in\', \'Lowest friction of all types of O-rings\', \'Twisting action disperses the pressure and minimizes power loss\', \'Maxim

In [19]:
from pathlib import Path

def make_file(start, end, filename):
    batch_file = Path(filename)
    # Create jsonl/ (or any other parent folder) before opening the file.
    batch_file.parent.mkdir(parents=True, exist_ok=True)
    safe_end = min(end, len(items))
    with batch_file.open("w", encoding="utf-8") as f:
        for i in range(start, safe_end):
            f.write(make_jsonl(items[i]))
            f.write("\n")
    print(f"Wrote {safe_end - start:,} local Qwen requests to {batch_file.resolve()}")
make_file(0, 1000, "../jsonl/0_1000.jsonl")

Wrote 1,000 local Qwen requests to C:\Users\aksha\PycharmProjects\PricerProject\jsonl\0_1000.jsonl


In [20]:
# Run the first saved request locally with qwen3.6:latest.
with open("../jsonl/0_1000.jsonl", encoding="utf-8") as f:
    request = json.loads(next(f))

assert request["model"] == "qwen3.6:latest"
response = ollama_chat_response(request["messages"])
response

{'model': 'qwen3.6:latest',
 'created_at': '2026-08-02T17:30:09.5178004Z',
 'message': {'role': 'assistant',
  'content': 'Title: DID 520 X-Ring Gold Chain - 98 Links\nCategory: Motorcycle Parts & Accessories\nBrand: DID\nDescription: This 98-link gold 520 X-Ring chain offers superior sealing and low friction for offroad applications.\nDetails: It features four-section sealing to keep dirt out and lubrication in, minimizing power loss through a twisting action that disperses pressure while providing maximum wear resistance.'},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 2054983600,
 'load_duration': 248095200,
 'prompt_eval_count': 277,
 'prompt_eval_duration': 172732000,
 'eval_count': 91,
 'eval_duration': 1629907000}

In [21]:
# Process the JSONL requests locally with qwen3.6:latest through Ollama.
from pathlib import Path

batch_module.MODEL = "qwen3.6:latest"
MODEL = batch_module.MODEL
requests_file = Path("../jsonl/0_1000.jsonl")
results_file = Path("../jsonl/batch_results.jsonl")

completed_ids = set()
if results_file.exists():
    with results_file.open(encoding="utf-8") as f:
        for line in f:
            try:
                completed_ids.add(str(json.loads(line)["custom_id"]))
            except (json.JSONDecodeError, KeyError):
                pass

with requests_file.open(encoding="utf-8") as requests, results_file.open("a", encoding="utf-8") as results:
    for line in tqdm(requests, desc="Running qwen3.6:latest locally", unit="item"):
        request = json.loads(line)
        custom_id = str(request["custom_id"])
        if custom_id in completed_ids:
            continue

        response = ollama_chat_response(request["messages"])
        summary = response["message"]["content"].strip()
        result = {"custom_id": custom_id, "summary": summary}
        results.write(json.dumps(result, ensure_ascii=False) + "\n")
        results.flush()
        items[int(custom_id)].summary = summary
        completed_ids.add(custom_id)

print(f"Completed {len(completed_ids):,} summaries with qwen3.6:latest")
print(items[0].summary)

Running qwen3.6:latest locally: 1000item [26:54,  1.61s/item]

Completed 1,000 summaries with qwen3.6:latest
Title: DID 520 X-Ring Gold Chain - 98 Links
Category: Motorcycle Parts & Accessories
Brand: DID
Description: This gold-colored 520 X-Ring chain features 98 links designed for offroad applications with superior sealing and low friction.
Details: It offers maximum wear resistance, keeps dirt out while retaining lubrication, and minimizes power loss through a twisting action that disperses pressure.


In [22]:
Batch.create(items, LITE_MODE)
Batch.run(workers=4)
Batch.fetch()
for index, item in enumerate(items):
    if not item.summary:
        print(index)
print(items[10234].summary)
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

Created 820 local Ollama batches using qwen3.6:latest


Local batches:   0%|          | 0/820 [04:18<?, ?batch/s]


KeyboardInterrupt: 

In [ ]:
username = "akshaymgupte87"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

In [25]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [26]:
LITE_MODE = False
##Taking it from others for now to avoid delay to process all records locally and push to huggingface.
# Will push to my own space later. This is the next independent cell to start from.
#Checkpoint 2
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

C:\Users\aksha\PycharmProjects\PricerProje\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\datasets--ed-donner--items_full. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  243MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.04MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [27]:
def random_pricer(item):
    return random.randrange(1,1000)
random.seed(42)
evaluate(random_pricer, test)

# That was fun!
# We can do better - here's another rather trivial model

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

evaluate(constant_pricer, test)

def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $1 $29 $690 $221 $52 $85 $540 $69 $187 $20 $180 $894 $36 $680 $572 $618 $689 $92 $580 $299 $91 $54 $79 $108 $214 $23 $597 $71 $495 $184 $674 $506 $639 $444 $389 $166 $404 $390 $247 $629 $811 $28 $523 $606 $139 $697 $401 $204 $83 $125 $91 $506 $662 $292 $29 $88 $350 $102 $356 $632 $275 $528 $161 $198 $45 $508 $26 $425 $52 $965 $928 $108 $62 $516 $265 $726 $634 $617 $888 $763 $359 $562 $98 $695 $12 $135 $372 $114 $746 $275 $13 $856 $204 $880 $124 $356 $128 $195 $76 $755 $257 $130 $241 $115 $97 $662 $126 $688 $661 $640 $350 $43 $496 $456 $6 $522 $727 $195 $65 $458 $300 $207 $782 $456 $645 $551 $126 $523 $194 $793 $650 $710 $28 $170 $827 $172 $755 $196 $273 $211 $182 $137 $916 $772 $549 $853 $226 $58 $185 $653 $406 $359 $766 $918 $380 $421 $91 $48 $93 $36 $737 $533 $539 $570 $742 $151 $389 $880 $588 $384 $349 $195 $119 $420 $357 $66 $681 $45 $789 $541 $132 $343 $85 $762 $675 $300 $544 $6 $368 $356 $560 $445 $403 $158 $915 $397 $602 $915 $14 

140.56967545


  0%|          | 0/200 [00:00<?, ?it/s]

$78 $25 $86 $71 $111 $89 $4 $75 $105 $189 $572 $238 $121 $86 $61 $108 $61 $91 $70 $22 $7 $17 $56 $34 $191 $312 $354 $121 $42 $61 $121 $81 $19 $60 $25 $678 $81 $85 $73 $103 $59 $61 $106 $114 $79 $116 $123 $109 $5 $61 $105 $11 $334 $21 $87 $6 $134 $101 $62 $129 $95 $63 $50 $31 $488 $51 $99 $304 $16 $65 $109 $124 $139 $122 $91 $105 $16 $131 $124 $122 $21 $129 $111 $42 $114 $81 $42 $165 $21 $95 $119 $46 $121 $106 $132 $88 $107 $17 $129 $434 $41 $24 $104 $2 $108 $23 $116 $259 $110 $158 $81 $174 $110 $12 $55 $29 $116 $121 $85 $38 $125 $52 $70 $25 $59 $81 $121 $42 $38 $2 $69 $3 $55 $111 $76 $126 $64 $71 $12 $2 $76 $109 $60 $121 $53 $109 $96 $369 $124 $108 $122 $35 $94 $1 $121 $138 $92 $85 $179 $91 $148 $115 $99 $128 $699 $118 $307 $91 $101 $131 $116 $119 $279 $118 $39 $8 $113 $48 $46 $48 $514 $116 $159 $108 $91 $119 $7 $74 $81 $114 $106 $90 $106 $2 $41 $61 $29 $140 $90 $115 

In [30]:
# Traditional Linear Regression!

np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

evaluate(linear_regression_pricer, test)
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])
regressor = LinearRegression()
regressor.fit(X, prices)

def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

evaluate(natural_language_linear_regression_pricer, test)
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

weight: 0.4486789805756249
weight_unknown: -6.627998877825423
text_length: 0.24694518955630418
Intercept: 51.11099822544196
Mean Squared Error: 25615.843488572416
R-squared Score: -0.05946814914318388


  0%|          | 0/200 [00:00<?, ?it/s]

$55 $29 $87 $90 $92 $60 $1 $78 $104 $192 $556 $228 $141 $81 $50 $124 $69 $83 $50 $8 $10 $9 $61 $2 $175 $298 $345 $114 $34 $55 $100 $77 $10 $38 $24 $676 $78 $94 $50 $90 $48 $55 $83 $102 $82 $119 $100 $106 $2 $52 $106 $9 $353 $28 $92 $26 $136 $110 $37 $121 $97 $69 $43 $17 $460 $41 $83 $288 $4 $86 $109 $101 $139 $113 $99 $103 $8 $122 $122 $111 $25 $107 $98 $10 $108 $63 $60 $164 $33 $82 $109 $41 $101 $94 $114 $104 $97 $9 $151 $426 $37 $11 $135 $5 $100 $25 $107 $277 $102 $182 $90 $169 $135 $11 $45 $38 $100 $119 $98 $21 $125 $72 $76 $32 $35 $60 $109 $61 $10 $2 $78 $12 $58 $95 $58 $106 $65 $65 $1 $8 $61 $110 $54 $107 $33 $91 $93 $364 $107 $92 $118 $44 $76 $19 $111 $122 $102 $61 $138 $76 $135 $116 $83 $113 $710 $120 $269 $101 $107 $125 $110 $124 $275 $101 $52 $4 $119 $26 $48 $53 $481 $141 $139 $64 $102 $116 $19 $77 $78 $103 $94 $99 $122 $1 $30 $65 $25 $133 $112 $113 

Number of selected words: 2000
Selected words: ['jack' 'jacket' 'jeep' 'jet' 'jigsaw' 'joint' 'joints' 'kawasaki'
 'keeping' 'keeps' 'key' 'keyboard' 'keypad' 'keys' 'kg' 'khz' 'kia'
 'kickstand' 'kids' 'king']


  0%|          | 0/200 [00:00<?, ?it/s]

$67 $124 $55 $6 $179 $211 $52 $50 $73 $7 $535 $186 $101 $150 $67 $96 $58 $49 $37 $29 $15 $87 $28 $9 $274 $241 $141 $13 $56 $80 $78 $57 $10 $43 $113 $418 $15 $65 $140 $71 $156 $61 $69 $26 $94 $59 $39 $49 $4 $13 $41 $91 $156 $31 $80 $69 $81 $166 $24 $12 $44 $18 $55 $24 $422 $93 $3 $312 $62 $212 $19 $33 $11 $130 $1 $36 $115 $36 $14 $130 $85 $59 $43 $42 $36 $117 $42 $105 $31 $177 $4 $106 $80 $27 $38 $91 $61 $1 $162 $102 $49 $56 $37 $15 $29 $4 $43 $214 $31 $98 $20 $20 $171 $42 $6 $149 $112 $25 $24 $40 $11 $108 $87 $28 $80 $27 $32 $127 $86 $29 $74 $40 $141 $23 $70 $55 $39 $103 $126 $76 $34 $171 $78 $95 $68 $91 $25 $235 $63 $32 $19 $189 $24 $27 $62 $49 $124 $28 $7 $9 $76 $26 $42 $13 $470 $22 $76 $50 $34 $9 $3 $23 $279 $27 $41 $144 $10 $40 $45 $144 $372 $18 $20 $13 $158 $32 $44 $16 $103 $36 $29 $122 $9 $27 $43 $60 $53 $6 $48 $18 

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [31]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])
evaluate(random_forest, test)
# This is how to save the model if you want to, particularly if you run this on a larger dataset

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

  0%|          | 0/200 [00:00<?, ?it/s]

$87 $74 $4 $18 $126 $151 $75 $72 $36 $222 $507 $287 $59 $63 $13 $0 $54 $88 $38 $51 $30 $24 $1 $25 $158 $310 $169 $128 $26 $51 $47 $92 $64 $20 $19 $513 $44 $35 $136 $65 $143 $47 $9 $60 $139 $30 $58 $58 $52 $11 $14 $53 $240 $38 $308 $39 $63 $161 $2 $45 $100 $5 $59 $2 $461 $3 $27 $195 $13 $137 $7 $20 $32 $131 $3 $75 $84 $21 $32 $91 $65 $73 $37 $51 $12 $31 $15 $112 $13 $96 $26 $175 $1 $82 $52 $51 $7 $55 $129 $286 $35 $2 $25 $66 $40 $29 $121 $192 $19 $175 $40 $63 $71 $34 $76 $51 $86 $16 $234 $25 $24 $28 $44 $46 $36 $23 $42 $57 $91 $2 $32 $48 $73 $30 $43 $54 $22 $4 $78 $44 $4 $159 $6 $105 $99 $58 $23 $313 $87 $2 $21 $130 $48 $72 $56 $77 $113 $22 $45 $7 $2 $8 $3 $31 $395 $41 $279 $30 $16 $42 $23 $12 $224 $25 $17 $66 $60 $0 $36 $9 $364 $22 $70 $91 $124 $46 $61 $77 $14 $30 $40 $57 $24 $6 $8 $22 $69 $45 $3 $16 

In [32]:
import xgboost as xgb
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$95 $91 $24 $12 $167 $111 $18 $20 $22 $105 $359 $201 $138 $131 $16 $16 $44 $13 $31 $48 $4 $74 $14 $46 $171 $262 $146 $35 $50 $39 $78 $86 $23 $22 $33 $303 $19 $48 $163 $82 $130 $63 $54 $56 $126 $31 $73 $48 $35 $13 $23 $74 $133 $5 $61 $91 $54 $137 $17 $12 $28 $10 $46 $1 $408 $91 $6 $251 $58 $140 $18 $41 $79 $145 $8 $74 $173 $19 $34 $69 $74 $54 $39 $25 $21 $65 $52 $132 $13 $183 $23 $91 $36 $20 $28 $75 $26 $39 $153 $158 $32 $42 $3 $55 $3 $48 $51 $248 $31 $70 $39 $18 $62 $67 $43 $23 $43 $25 $53 $58 $59 $147 $86 $20 $86 $25 $5 $114 $136 $15 $27 $50 $90 $24 $72 $89 $84 $38 $43 $59 $15 $130 $40 $64 $49 $67 $37 $282 $77 $27 $19 $162 $22 $53 $30 $119 $128 $16 $14 $5 $99 $26 $5 $9 $391 $2 $112 $50 $16 $10 $29 $16 $197 $27 $30 $49 $27 $0 $31 $24 $253 $14 $189 $19 $157 $76 $56 $73 $21 $34 $89 $68 $1 $56 $10 $59 $12 $9 $36 $23 

In [ ]:
# Train a Random Forest on the complete training dataset.
# This can take many hours for the 800,000-item full dataset.
from sklearn.ensemble import RandomForestRegressor

if X.shape[0] != len(prices):
    raise ValueError("X and prices must contain the same number of training rows")

full_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
full_rf_model.fit(X, prices)

def full_random_forest_pricer(item):
    item_text = item.summary or ""
    item_features = vectorizer.transform([item_text])
    prediction = full_rf_model.predict(item_features)[0]
    return float(np.clip(prediction, 0, 1000))

evaluate(full_random_forest_pricer, test)

# Optional: save the trained model and its matching vectorizer.
# import joblib
# joblib.dump({"model": full_rf_model, "vectorizer": vectorizer}, "full_random_forest.joblib")

In [ ]:
### Starting new section


In [4]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [5]:
# Write the test set to a CSV

with open('../human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])
# Read it back in

human_predictions = []
with open('../human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")
evaluate(human_pricer, test, size=100)

Human predicted 120.0 for an item that actually costs 219.0


  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

In [6]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize

        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)
evaluate(neural_network, test)

Number of trainable parameters: 669,249


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 12764.440, Val Loss: 12143.647


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 12461.828, Val Loss: 11167.079


  0%|          | 0/200 [00:00<?, ?it/s]

$150 $31 $27 $191 $42 $134 $71 $96 $22 $166 $213 $126 $36 $96 $10 $10 $10 $19 $3 $1 $32 $57 $14 $23 $216 $116 $247 $37 $135 $47 $76 $156 $69 $1 $31 $393 $28 $99 $76 $72 $159 $14 $30 $16 $20 $61 $29 $27 $68 $31 $34 $19 $242 $21 $110 $126 $29 $78 $6 $66 $85 $0 $31 $5 $387 $19 $48 $307 $30 $85 $13 $25 $108 $103 $0 $56 $57 $13 $13 $80 $73 $59 $40 $49 $19 $242 $28 $89 $51 $115 $23 $10 $1 $6 $48 $44 $25 $15 $67 $253 $4 $27 $21 $59 $38 $47 $70 $298 $1 $163 $34 $7 $72 $24 $56 $106 $29 $66 $56 $109 $35 $188 $79 $1 $99 $43 $18 $71 $38 $4 $37 $33 $62 $20 $29 $17 $81 $2 $8 $35 $7 $99 $35 $53 $31 $40 $36 $175 $115 $2 $13 $60 $2 $19 $36 $123 $85 $3 $56 $14 $27 $6 $4 $24 $269 $13 $90 $34 $20 $54 $52 $13 $249 $33 $15 $59 $28 $11 $28 $42 $24 $10 $4 $0 $54 $42 $67 $124 $53 $28 $62 $26 $2 $111 $2 $42 $152 $43 $6 $17 

In [7]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

LOCAL_QWEN_MODEL = "qwen3.6:latest"
LOCAL_OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
if not LOCAL_OLLAMA_HOST.startswith(("http://", "https://")):
    LOCAL_OLLAMA_HOST = f"http://{LOCAL_OLLAMA_HOST}"

def build_price_estimation_messages(item):
    prompt = f"Estimate the price of this product. Respond with the price only, no explanation.\n\n{item.summary}"
    return [{"role": "user", "content": prompt}]

def call_local_qwen(messages):
    payload = {
        "model": LOCAL_QWEN_MODEL,
        "messages": messages,
        "stream": False,
        "think": False,
        "keep_alive": "30m",
        "options": {"temperature": 0, "num_predict": 20, "seed": 42},
    }
    request = Request(
        f"{LOCAL_OLLAMA_HOST}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=300) as response:
            result = json.load(response)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(f"Cannot reach Ollama at {LOCAL_OLLAMA_HOST}. Start Ollama first.") from exc
    return result["message"]["content"].strip()

def local_qwen_3_6_pricer(item):
    return call_local_qwen(build_price_estimation_messages(item))

print(test[0].summary)
print(f"Local Qwen prediction: {local_qwen_3_6_pricer(test[0])}")
print(f"Actual price: ${test[0].price:,.2f}")
evaluate(local_qwen_3_6_pricer, test, workers=1)

# Hosted frontier-model examples are retained for reference but disabled.
# def gpt_4_1_nano_pricer(item):
#     response = completion(model="openai/gpt-4.1-nano", messages=build_price_estimation_messages(item))
#     return response.choices[0].message.content
# evaluate(gpt_4_1_nano_pricer, test)

# def claude_opus_4_5_pricer(item):
#     response = completion(model="anthropic/claude-opus-4-5", messages=build_price_estimation_messages(item))
#     return response.choices[0].message.content
# evaluate(claude_opus_4_5_pricer, test)

# def gemini_3_pro_preview_pricer(item):
#     response = completion(model="gemini/gemini-3-pro-preview", messages=build_price_estimation_messages(item), reasoning_effort="low")
#     return response.choices[0].message.content
# evaluate(gemini_3_pro_preview_pricer, test, size=50, workers=2)

# def gemini_3_1_flash_lite_pricer(item):
#     response = completion(model="gemini/gemini-3.1-flash-lite", messages=build_price_estimation_messages(item))
#     return response.choices[0].message.content
# evaluate(gemini_3_1_flash_lite_pricer, test)

# def grok_4_1_fast_pricer(item):
#     response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=build_price_estimation_messages(item), seed=42)
#     return response.choices[0].message.content
# evaluate(grok_4_1_fast_pricer, test)

# def gpt_5_1_pricer(item):
#     response = completion(model="gpt-5.1", messages=build_price_estimation_messages(item), reasoning_effort="high", seed=42)
#     return response.choices[0].message.content
# evaluate(gpt_5_1_pricer, test)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.
Local Qwen prediction: $249
Actual price: $219.00


  0%|          | 0/200 [00:00<?, ?it/s]

$30 $26 $39 $40 $20 $185 $54 $85 $20 $245 $263 $20 $6 $29 $49 $17 $11 $20 $140 $93 $99 $34 $5 $26 $182 $154 $245 $5 $51 $55 $6 $20 $140 $64 $25 $120 $25 $30 $64 $19 $174 $55 $22 $165 $70 $1 $1 $2 $55 $112 $22 $117 $125 $0 $37 $49 $9 $90 $12 $4 $146 $48 $65 $80 $329 $5 $150 $355 $35 $44 $22 $9 $20 $3 $35 $10 $39 $10 $1 $6 $50 $4 $14 $83 $8 $20 $8 $126 $50 $21 $4 $49 $4 $15 $3 $138 $4 $111 $140 $185 $10 $77 $17 $11 $51 $33 $16 $355 $6 $200 $34 $124 $4 $82 $54 $29 $1 $10 $16 $347 $10 $210 $40 $36 $110 $15 $7 $31 $89 $49 $79 $13 $5 $14 $15 $2 $120 $20 $82 $18 $19 $49 $35 $9 $104 $58 $5 $160 $135 $13 $4 $84 $21 $50 $7 $189 $17 $41 $30 $25 $110 $17 $8 $0 $141 $7 $149 $34 $10 $3 $9 $6 $20 $7 $56 $59 $2 $37 $49 $48 $45 $5 $150 $159 $5 $18 $83 $3 $25 $11 $5 $19 $19 $111 $60 $70 $20 $10 $35 $13 

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: C:\Users\aksha\PycharmProjects\PricerProje\venv\Scripts\python.exe
PyTorch: 2.12.1+cu130
PyTorch CUDA build: 13.0
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce RTX 3090


In [9]:
# Local Ollama evaluation only -- no Hugging Face model download and no fine-tuning.

import numpy as np

from pricer.evaluator import Tester

OLLAMA_EVALUATION_ITEMS = 20000

if "test" not in globals():
    raise RuntimeError("Run the dataset-loading cell before this evaluation cell.")
if "local_qwen_3_6_pricer" not in globals():
    raise RuntimeError("Run the preceding local Ollama Qwen cell first.")

evaluation_size = min(OLLAMA_EVALUATION_ITEMS, len(test))
print(f"Evaluating installed Ollama model qwen3.6:latest on {evaluation_size:,} held-out items...")
ollama_qwen_tester = Tester(
    local_qwen_3_6_pricer,
    test,
    title="Local Ollama qwen3.6:latest (no fine-tuning)",
    size=evaluation_size,
    workers=1,
)
ollama_qwen_tester.run()
ollama_qwen_mae = float(np.mean(ollama_qwen_tester.errors))

reference_results = {
    "OpenAI mini, 200 fine-tuning examples": 96.58,
    "OpenAI mini, 2,000 fine-tuning examples": 79.29,
    "OpenAI nano, 2,000 fine-tuning examples": 82.26,
    "OpenAI nano, 20,000 fine-tuning examples": 67.75,
    "Local Ollama qwen3.6:latest, 2000 fine tuning examples no fine-tuning: $57.33 MAE"
    "Local Ollama qwen3.6:latest, no fine-tuning": ollama_qwen_mae,
}
print("\n# Historical-reference comparison (lower MAE is better)")
print(f"# Local Qwen was evaluated on {evaluation_size:,} held-out items through Ollama.")
print("# No adapter was trained and no Hugging Face model weights were downloaded.")
for label, mae in sorted(reference_results.items(), key=lambda result: result[1]):
    print(f"# {label}: ${mae:,.2f} MAE")

# OpenAI managed fine-tuning remains disabled; the values above are historical references only.
# Local LoRA/QLoRA fine-tuning is intentionally deferred to a future section.

Evaluating installed Ollama model qwen3.6:latest on 10,000 held-out items...


  0%|          | 0/10000 [00:00<?, ?it/s]

$30 $26 $39 $40 $20 $185 $54 $85 $20 $245 $263 $20 $6 $29 $49 $17 $11 $20 $140 $93 $99 $34 $5 $26 $182 $154 $245 $5 $51 $55 $6 $20 $140 $64 $25 $120 $25 $30 $64 $19 $174 $55 $22 $165 $70 $1 $1 $2 $55 $112 $22 $117 $125 $0 $37 $49 $9 $90 $12 $4 $146 $48 $65 $80 $329 $5 $150 $355 $35 $44 $22 $9 $20 $3 $35 $10 $39 $10 $1 $6 $50 $4 $14 $83 $8 $20 $8 $126 $50 $21 $4 $49 $4 $15 $3 $138 $4 $111 $140 $185 $10 $77 $17 $11 $51 $33 $16 $355 $6 $200 $34 $124 $4 $82 $54 $29 $1 $10 $16 $347 $10 $210 $40 $36 $110 $15 $7 $31 $89 $49 $79 $13 $5 $14 $15 $2 $120 $20 $82 $18 $19 $49 $35 $9 $104 $58 $5 $160 $135 $13 $4 $84 $21 $50 $7 $189 $17 $41 $30 $25 $110 $17 $8 $0 $141 $7 $149 $34 $10 $3 $9 $6 $20 $7 $56 $59 $2 $37 $49 $48 $45 $5 $150 $159 $5 $18 $83 $3 $25 $11 $5 $19 $19 $111 $60 $70 $20 $10 $35 $13 $9 $43 $15 $11 $2 $11 $15 $170 $4 $81 $155 $300 $100 $1 $20 $9 $11 $18 $1 $2 $25 $35 $70 $167 $101 $101 $7 $3 $51 $59 $10 $7 $43 $84 $20 $10 $90 $1 $92 $70 $10 $1 $106 $16 $51 $45 $4 $10 $3 $2 $45 $349 $2

KeyboardInterrupt: 

In [ ]:
# imports
### this is for openai . Ignored for now
import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")
openai = OpenAI()
# Data size
# OpenAI recommends fine-tuning with a small population of 50-100 examples
#
#

# OpenAI recommends fine-tuning with populations of 50-100 examples


fine_tune_train = train[:100]
fine_tune_validation = val[:50]
len(fine_tune_train)
# Step 1
# Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]
messages_for(fine_tune_train[0])
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()
print(make_jsonl(train[:3]))
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
train_file
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
validation_file
# https://platform.openai.com/storage/files/
#
# Step 2
# And now time to Fine-tune!
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)
openai.fine_tuning.jobs.list(limit=1)
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
job_id
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data
# https://platform.openai.com/finetune
#
# Step 3
# Test our fine tuned model

fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model_name
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]
# Try this out

test_messages_for(test[0])
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))
evaluate(gpt_4__1_nano_fine_tuned, test)

In [ ]:
#Trying Qlora

In [1]:
# imports

import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime

W0803 10:14:12.675000 32876 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

LITE_MODE = False

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"


FINETUNED_MODEL = f"ed-donner/price-2025-11-30_15.10.55-lite"

In [4]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [1]:
!uv run price_is_right.py

Uninstalled 1 package in 18ms
Installed 1 package in 28ms
error: Failed to spawn: `price_is_right.py`
  Caused by: program not found
